# clue_train

> The clue board ready for RL training (Phase 11, Part D): batched episodes stepped one agent-turn at a time for a shared policy, individual, team and credit rewards, trajectory exports (per-agent steps and chat format) from turn logs, and a scripted teacher policy for supervised warm-starting. Nothing is trained here.

In [ ]:
#| default_exp clue_train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import heapq
import json
from pathlib import Path

import numpy as np

from explRL.clue_agents import ClueBoard, ClueLead, ClueMessages, RuleAgent
from explRL.clue_text import TextClueEnv
from explRL.clueboard import ClueBoardEnv
from explRL.multi import spawnRngs

BOARD = ('m', 'r', 'R', 'value', 'life', 'penalty')


def _channel(name, N, seed, passPooled=False):
    return dict(none=None, blackboard=ClueBoard(), hierarchical=ClueLead(N, passPooled),
                messages=ClueMessages(N, 2, spawnRngs(seed + 2000, N)))[name]


class ClueEpisodes:
    """A batch of clue boards stepped one agent-turn at a time (event order per board), for rollouts with a shared
    policy: `reset()` and `step()` return, per board, whose turn it is and that agent's text observation.
    `seeds`: a board seed per batch entry (repeats allowed; default seed + b); `scripted`: {slot: factory(channel, rng)}
    for members that act on the board directly (liars; rngs as runClue's); `text`: TextClueEnv options."""
    def __init__(self, batch, nAgents, settings=None, scheme='individual', turns=40, seed=0, channel='blackboard',
                 seeds=None, scripted=None, text=None, passPooled=False):
        self.batch, self.N, self.turns, self.seed, self.channelName = int(batch), int(nAgents), int(turns), int(seed), channel
        self.settings, self.scheme = {k: v for k, v in (settings or {}).items() if k in BOARD}, scheme
        self.seeds = [int(s) for s in seeds] if seeds is not None else [self.seed + b for b in range(self.batch)]
        if len(self.seeds) != self.batch:
            raise ValueError('one seed per board')
        self.scripted, self.text, self.passPooled = dict(scripted or {}), dict(text or {}), bool(passPooled)

    def reset(self):
        self.boards = [ClueBoardEnv(s, self.N, scheme=self.scheme, **self.settings) for s in self.seeds]
        self.channels = [_channel(self.channelName, self.N, s, self.passPooled) for s in self.seeds]
        self.envs = [TextClueEnv(bd, **self.text) for bd in self.boards]
        self.members = [self._members(b) for b in range(self.batch)]
        self.queues = [[(0.0, i) for i in range(self.N)] for _ in range(self.batch)]
        self.done, self.obs, self.last = [[0] * self.N for _ in range(self.batch)], [None] * self.batch, [None] * self.batch
        return [self._next(b) for b in range(self.batch)]

    def _members(self, b):
        slots = sorted(self.scripted)
        out = {}
        for slot, rng in zip(slots, spawnRngs(self.seeds[b] + 3000, len(slots)) if slots else []):
            out[slot] = self.scripted[slot](self.channels[b], rng)
            out[slot].id = slot
        return out

    def _requeue(self, b, i, t):
        self.done[b][i] += 1
        if self.done[b][i] < self.turns:
            heapq.heappush(self.queues[b], (t, i))

    def _next(self, b):
        "(agent, observation) of board b's next text turn (scripted turns run on the way), or None when done."
        q = self.queues[b]
        while q and q[0][1] in self.members[b]:
            t, i = heapq.heappop(q)
            self._requeue(b, i, t + self.members[b][i].event(self.boards[b], t).dt)
        if not q:
            return None
        t, i = q[0]
        self.obs[b] = self.envs[b].observe(i, t, self.channels[b])
        return i, self.obs[b]

    def step(self, replies):
        "replies: per board, the acting agent's JSON reply (dict) or None. -> (next, rewards [(agent, r)], done flags)."
        rewards = []
        for b, reply in enumerate(replies):
            if not self.queues[b]:
                rewards.append(None)
                continue
            t, i = heapq.heappop(self.queues[b])
            env = self.envs[b]
            action, report, valid = env.parse(reply, i)
            r, dt, _, info = self.boards[b].apply(i, action, t)
            env.post(i, report, t, self.channels[b], info)
            self.last[b] = dict(agent=i, t=t, obs=self.obs[b], action=action, report=report, valid=valid, reward=r,
                                info=info, explore=bool(env.explored(i, action, info)),
                                summary=env.summary(i, t, action, r, report, info))
            rewards.append((i, r))
            self._requeue(b, i, t + dt)
        return [self._next(b) for b in range(self.batch)], rewards, [not q for q in self.queues]


def exportTrajectories(log, path, format='jsonl', system=''):
    """A run's turn log (turns.jsonl records) -> per-agent steps (`jsonl`: key, agent, t, obs, action, reward, done) or
    chat records (`chat`: system / user / assistant messages and the reward). Returns the path."""
    path, recs = Path(path), [x for x in log if x.get('kind', 'turn') == 'turn']
    last = {}
    for n, x in enumerate(recs):
        last[(x.get('key'), x['agent'])] = n
    out = []
    for n, x in enumerate(recs):
        done = last[(x.get('key'), x['agent'])] == n
        if format == 'chat':
            out.append(dict(messages=[dict(role='system', content=system), dict(role='user', content=x['obs']),
                                      dict(role='assistant', content=x['response'])], reward=x['reward'], done=done))
        else:
            out.append(dict(key=x.get('key'), agent=x['agent'], t=x['t'], obs=x['obs'], action=x['response'],
                            reward=x['reward'], done=done))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(json.dumps(r, sort_keys=True) + '\n' for r in out))
    return path


def readTrajectories(path):
    return [json.loads(x) for x in Path(path).read_text().splitlines()]


def teacherAction(board, i, channel, rng=None, share='needed'):
    "The rule agent's choice for agent i now, rendered as the JSON reply the text interface parses (SFT targets)."
    agent = RuleAgent(channel, rng if rng is not None else np.random.default_rng(i), share, 0.0)
    agent.id = i
    kind, arg = agent.choose(board, board.tWorld)
    reply = dict(action=kind, target='', answer=0, clue='', notes='teacher')
    if kind == 'open':
        reply['target'] = f'T{arg}'
    elif kind == 'submit':
        reply['answer'] = int(arg)
    elif kind == 'share':
        k, j, r = arg
        reply.update(target=f'T{k}', clue=f'x mod {board.tasks[k].task.primes[j]} = {r}')
    return reply

## Tests

In [ ]:
import json, tempfile
from pathlib import Path
from explRL.clue_train import *
from explRL.clue_runs import runClue
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import LLMConfig

S = dict(m=3, r=1, R=105)
def rollout(seed, N=4, batch=2, turns=12, channel='blackboard', scheme='individual'):
    ep = ClueEpisodes(batch, N, S, scheme=scheme, turns=turns, seed=seed, channel=channel)
    nxt, total = ep.reset(), []
    while any(x is not None for x in nxt):
        replies = [teacherAction(ep.boards[b], x[0], ep.channels[b]) if x is not None else None for b, x in enumerate(nxt)]
        nxt, rew, done = ep.step(replies)
        total.append(rew)
    return ep, total
# deterministic per seed; every agent makes its turns; done flags at the end
e1, r1 = rollout(3); e2, r2 = rollout(3)
assert r1 == r2 and all(sum(d) == 4 * 12 for d in e1.done)
assert all(isinstance(x[0], int) for step in r1 for x in step if x)
# the teacher solves every subtask it holds all clues for (one agent holds all), never answering wrong
solo, _ = rollout(5, N=1, batch=3, turns=30, channel='none')
for bd in solo.boards:
    assert bd.counts['solved'] >= bd.counts['opens'] - 1 and bd.counts['wrong'] == 0 and bd.counts['solved'] > 0
# with a channel the teacher shares, and a team solves assisted subtasks
team, _ = rollout(7, N=4, batch=2, turns=30)
assert all(bd.counts['shares'] > 0 for bd in team.boards) and any(a for bd in team.boards for *_, a in bd.solveLog)
# reward schemes: same team total delivered plus pending
tot = {}
for sc in ('individual', 'team', 'credit'):
    ep, rw = rollout(9, scheme=sc)
    tot[sc] = sum(r for step in rw for x in step if x for r in [x[1]]) + sum(bd.pending.sum() for bd in ep.boards)
assert abs(tot['individual'] - tot['team']) < 1e-9 and abs(tot['individual'] - tot['credit']) < 1e-9
# exports from a run's turn log round-trip, in both formats
rows, log = runClue(LLMConfig(task='clue', channel='blackboard', N=3, turns=4, world=S), 900, LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
log = [dict(key='k1', **x) for x in log]
tmp = Path(tempfile.mkdtemp())
steps = readTrajectories(exportTrajectories(log, tmp / 'steps.jsonl'))
assert len(steps) == 12 and sum(s['done'] for s in steps) == 3 and steps[0]['obs'] == log[0]['obs'] and steps[0]['action'] == log[0]['response']
chat = readTrajectories(exportTrajectories(log, tmp / 'chat.jsonl', 'chat', system='rules'))
assert [m['role'] for m in chat[0]['messages']] == ['system', 'user', 'assistant'] and chat[0]['messages'][0]['content'] == 'rules'
assert readTrajectories(tmp / 'chat.jsonl') == chat

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()